## 1. Định nghĩa Vấn đề
**Mô tả**
Bộ dữ liệu housing (Boston-style) gồm các mẫu đại diện cho các khu vực địa phương; mỗi mẫu chứa các đặc trưng về điều kiện kinh tế, hạ tầng và đặc tính nhà ở. Mục tiêu là dự đoán giá nhà trung vị của từng khu vực dựa trên các đặc trưng này. File dữ liệu trong dự án: housing.csv.
**Dữ liệu vào (Features)**

+ CRIM — Tỷ lệ tội phạm theo khu vực
+ ZN — Tỷ lệ đất dành cho khu dân cư >25.000 sq.ft
+ INDUS — Tỷ lệ diện tích thương mại/phi bán lẻ (%)
+ CHAS — Biến chỉ số sông Charles (1 nếu giáp sông, 0 nếu không)
+ NOX — Nồng độ NOx (phần triệu)
+ RM — Số phòng trung bình trên mỗi cư trú
+ AGE — Tỷ lệ nhà xây trước năm 1940 (%)
+ DIS — Khoảng cách đến các trung tâm lao động (trọng số)
+ RAD — Chỉ số tiếp cận đường vành đai/xa lộ
+ TAX — Thuế tài sản hàng năm (trên 10.000$)
+ PTRATIO — Tỉ lệ học sinh/giáo viên
+ B — 1000(Bk - 0.63)^2 (chỉ số dân cư)
+ LSTAT — % dân có thu nhập thấp

**Kết quả / Mục tiêu (Target / Goal)**

+ MEDV — Giá nhà trung vị (median value) — bài toán hồi quy (dự đoán giá liên tục).

## 2. Chuẩn bị vấn đề (Prepare Problem)

### 2.1 Tải thư viện

In [1]:
# Load libraries
import numpy
import pandas as pd
from numpy import arange
from matplotlib import pyplot
from pandas import read_csv
from pandas import set_option
from pandas.plotting import scatter_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error

### 2.1. Nạp dữ liệu (Load Dataset)

In [2]:
# Load dataset
filename = '../data/raw/housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataset = pd.read_csv(filename, names=names, index_col=0)

In [3]:
# Split-out validation dataset
array = dataset.values
X = array[:,0:13]
Y = array[:,13]
validation_size = 0.20
seed = 7
X_train, X_validation, Y_train, Y_validation = train_test_split(X, Y,
test_size=validation_size, random_state=seed)

## 6. Hoàn thiện Mô hình

### 6.1 Dự đoán trên tập dữ liệu kiểm định

In [5]:
# prepare the model
scaler = StandardScaler().fit(X_train)
rescaledX = scaler.transform(X_train)
model = GradientBoostingRegressor(random_state=seed, n_estimators=400)
model.fit(rescaledX, Y_train)

GradientBoostingRegressor(n_estimators=400, random_state=7)

In [6]:
# transform the validation dataset
rescaledValidationX = scaler.transform(X_validation)
predictions = model.predict(rescaledValidationX)
print(mean_squared_error(Y_validation, predictions))

11.902132586880027


### 6.2 Tái sử dụng Model từ Exp Folder

In [8]:
import sys, os
from datetime import datetime
sys.path.insert(0, os.path.abspath("../src"))
from save_model import load_best_model

# Danh sách các model đã lưu
models_source = [
    ("../exp/27-10-2025_improve_accuracy", "Improve_Accuracy (ScaledGBM_Tuned)"),
    ("../exp/27-10-2025_housing_regression", "Evaluate_Algorithms (ScaledKNN)")
]

print("\n" + "=" * 70)
print("AVAILABLE SAVED MODELS IN EXP FOLDER")
print("=" * 70)

for i, (folder, desc) in enumerate(models_source, 1):
    exists = os.path.exists(folder)
    status = " Available" if exists else " Not found"
    print(f"{i}. {desc}")
    print(f"   Path: {folder}")
    print(f"   Status: {status}\n")



AVAILABLE SAVED MODELS IN EXP FOLDER
1. Improve_Accuracy (ScaledGBM_Tuned)
   Path: ../exp/27-10-2025_improve_accuracy
   Status:  Available

2. Evaluate_Algorithms (ScaledKNN)
   Path: ../exp/27-10-2025_housing_regression
   Status:  Not found



In [21]:
# Load model từ exp folder
# 1. Improved model (ScaledGBM_Tuned)
model_improved_folder = "../exp/27-10-2025_improve_accuracy"
model_baseline_folder = "../exp/27-10-2025_model_baseline"

print("=" * 70)
print(f"LOADING MODELS FROM EXP FOLDER")
print("=" * 70)

model_loaded = None
scaler_loaded = None
model_baseline = None
scaler_baseline = None

# Load improved model
try:
    loaded_data = load_best_model(model_improved_folder)
    model_loaded = loaded_data['model']
    scaler_loaded = loaded_data['scaler']
    print(f"\n Improved Model (ScaledGBM_Tuned) loaded successfully!")
    print(f"  Path: {model_improved_folder}")
except FileNotFoundError as e:
    print(f"\n Improved model not found: {e}")

# Load baseline model
try:
    loaded_data_baseline = load_best_model(model_baseline_folder)
    model_baseline = loaded_data_baseline['model']
    scaler_baseline = loaded_data_baseline['scaler']
    print(f"\n Baseline Model loaded successfully!")
    print(f"  Path: {model_baseline_folder}")
except FileNotFoundError as e:
    print(f"\n Baseline model not found: {e}")
    print("  (Will skip baseline comparison)")

print("=" * 70)


LOADING MODELS FROM EXP FOLDER
✓ MODEL LOADED SUCCESSFULLY
Model: GradientBoostingRegressor
Scaler: StandardScaler

Paths:
  - Model: ../exp/27-10-2025_improve_accuracy\best_model.joblib
  - Scaler: ../exp/27-10-2025_improve_accuracy\scaler.joblib

 Improved Model (ScaledGBM_Tuned) loaded successfully!
  Path: ../exp/27-10-2025_improve_accuracy
✓ MODEL LOADED SUCCESSFULLY
Model: LinearRegression
Scaler: StandardScaler

Paths:
  - Model: ../exp/27-10-2025_model_baseline\best_model.joblib
  - Scaler: ../exp/27-10-2025_model_baseline\scaler.joblib

 Baseline Model loaded successfully!
  Path: ../exp/27-10-2025_model_baseline


In [20]:
# Dự đoán với cả improved và baseline model
print("\n" + "=" * 70)
print("MAKING PREDICTIONS")
print("=" * 70)

# Improved model
if model_loaded is not None and scaler_loaded is not None:
    print("\n6.2a - Improved Model (ScaledGBM_Tuned)")
    print("-" * 70)
    
    rescaledValidationX_loaded = scaler_loaded.transform(X_validation)
    predictions_loaded = model_loaded.predict(rescaledValidationX_loaded)
    
    from sklearn.metrics import mean_absolute_error
    mse_loaded = mean_squared_error(Y_validation, predictions_loaded)
    rmse_loaded = numpy.sqrt(mse_loaded)
    mae_loaded = mean_absolute_error(Y_validation, predictions_loaded)
    
    print(f"Validation Metrics:")
    print(f"  MSE:  {mse_loaded:.4f}")
    print(f"  RMSE: {rmse_loaded:.4f}")
    print(f"  MAE:  {mae_loaded:.4f}")
else:
    print(" Improved model not loaded")
    rmse_loaded = None

# Baseline model
if model_baseline is not None and scaler_baseline is not None:
    print("\n6.2b - Baseline Model")
    print("-" * 70)
    
    rescaledValidationX_baseline = scaler_baseline.transform(X_validation)
    predictions_baseline = model_baseline.predict(rescaledValidationX_baseline)
    
    from sklearn.metrics import mean_absolute_error
    mse_baseline = mean_squared_error(Y_validation, predictions_baseline)
    rmse_baseline = numpy.sqrt(mse_baseline)
    mae_baseline = mean_absolute_error(Y_validation, predictions_baseline)
    
    print(f"Validation Metrics:")
    print(f"  MSE:  {mse_baseline:.4f}")
    print(f"  RMSE: {rmse_baseline:.4f}")
    print(f"  MAE:  {mae_baseline:.4f}")
else:
    print(" Baseline model not loaded")
    rmse_baseline = None

print("\n" + "=" * 70)



MAKING PREDICTIONS

6.2a - Improved Model (ScaledGBM_Tuned)
----------------------------------------------------------------------
Validation Metrics:
  MSE:  11.9021
  RMSE: 3.4499
  MAE:  2.1542

6.2b - Baseline Model
----------------------------------------------------------------------
Validation Metrics:
  MSE:  34.0565
  RMSE: 5.8358
  MAE:  3.7808



### 6.3 So sánh Kết quả

In [ ]:
# So sánh kết quả
print("\n" + "=" * 70)
print("IMPROVEMENT ANALYSIS")
print("=" * 70)

if rmse_loaded is not None and rmse_baseline is not None:
    print(f"\nBaseline Model RMSE:  {rmse_baseline:.4f}")
    print(f"Improved Model RMSE:  {rmse_loaded:.4f}")
    
    improvement = ((rmse_baseline - rmse_loaded) / rmse_baseline) * 100
    
    print("\n" + "-" * 70)
    if improvement > 0:
        print(f" IMPROVEMENT: {improvement:.2f}% better with tuned model!")
        print(f"  Reduced RMSE by: {rmse_baseline - rmse_loaded:.4f}")
    elif improvement < 0:
        print(f" REGRESSION: {abs(improvement):.2f}% worse")
        print(f"  Increased RMSE by: {rmse_loaded - rmse_baseline:.4f}")
    else:
        print(f"= NO CHANGE: Both models have same RMSE")
    print("-" * 70)
    
    print(f"\nMetrics Comparison:")
    print(f"{'':25} | {'Baseline':>15} | {'Improved':>15} | {'Difference':>15}")
    print(f"{'-'*25}-+-{'-'*15}-+-{'-'*15}-+-{'-'*15}")
    print(f"{'RMSE':25} | {rmse_baseline:>15.4f} | {rmse_loaded:>15.4f} | {rmse_baseline-rmse_loaded:>15.4f}")
    if 'mae_baseline' in locals() and 'mae_loaded' in locals():
        print(f"{'MAE':25} | {mae_baseline:>15.4f} | {mae_loaded:>15.4f} | {mae_baseline-mae_loaded:>15.4f}")
    
elif rmse_loaded is not None:
    print(f"\nImproved Model RMSE: {rmse_loaded:.4f}")
    print("(Baseline model not available for comparison)")
    
print("\n" + "=" * 70)



IMPROVEMENT ANALYSIS

Baseline Model RMSE:  5.8358
Improved Model RMSE:  3.4499

----------------------------------------------------------------------
 IMPROVEMENT: 40.88% better with tuned model!
  Reduced RMSE by: 2.3858
----------------------------------------------------------------------

Metrics Comparison:
                          |        Baseline |        Improved |      Difference
--------------------------+-----------------+-----------------+----------------
RMSE                      |          5.8358 |          3.4499 |          2.3858
MAE                       |          3.7808 |          2.1542 |          1.6265

